In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text('environment', 'dev')
dbutils.widgets.text('catalog_name', 'naturapet_dev')
dbutils.widgets.text('storage_account', 'demodldb')
dbutils.widgets.text('storage_container', 'democodex')
dbutils.widgets.text('data_year', '2026')
dbutils.widgets.text('domain_name', 'comercial')

environment = dbutils.widgets.get('environment').strip().lower()
catalog = dbutils.widgets.get('catalog_name').strip()
storage_account = dbutils.widgets.get('storage_account').strip()
storage_container = dbutils.widgets.get('storage_container').strip()
data_year = dbutils.widgets.get('data_year').strip()
domain = dbutils.widgets.get('domain_name').strip().lower()

bronze_schema = f'{domain}_bronze'
silver_schema = f'{domain}_silver'
storage_uri = f'abfss://{storage_container}@{storage_account}.dfs.core.windows.net'
silver_root = f'{storage_uri}/external/{environment}/{domain}/silver/{data_year}'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{silver_schema}`')

DATE_COLUMNS = {
    'fecha', 'fecha_apertura', 'fecha_alta', 'fecha_registro', 'fecha_venta',
    'fecha_devolucion', 'fecha_compra', 'fecha_ajuste', 'fecha_solicitud',
    'fecha_prometida', 'fecha_entrega', 'fecha_inicio', 'fecha_fin',
    'fecha_ingreso', 'fecha_generacion'
}
DECIMAL_18_2_COLUMNS = {
    'precio_lista', 'costo_estandar', 'valor_bruto', 'descuento_total',
    'iva_total', 'valor_total', 'costo_total', 'margen_bruto',
    'precio_unitario', 'descuento_valor', 'base_sin_iva', 'iva_valor',
    'valor_total_linea', 'costo_total_linea', 'margen_bruto_linea',
    'valor_reintegrado', 'gasto_marketing', 'ventas_atribuidas_valor',
    'valor_impacto_costo', 'costo_unitario', 'subtotal_sin_iva',
    'valor_total_compra', 'valor_inventario_costo', 'monto_presupuestado',
    'monto_real', 'monto_presupuesto', 'monto_ejecutado',
    'variacion_presupuesto', 'meta_ventas', 'meta_margen_bruto',
    'salario_base_mensual', 'costo_negociado', 'costo_base_envio',
    'costo_envio'
}
DECIMAL_10_4_COLUMNS = {
    'iva_tasa', 'descuento_pct', 'roas', 'costo_transaccion_pct',
    'margen_objetivo', 'peso_demanda', 'factor_demanda',
    'calificacion_servicio', 'calificacion_promedio', 'descuento_base_pct',
    'meta_tasa_devolucion'
}
BOOLEAN_COLUMNS = {
    'es_fin_de_semana', 'es_festivo_colombia', 'es_ciudad_principal',
    'permite_domicilio', 'atiende_b2b', 'controla_inventario',
    'consentimiento_marketing', 'cliente_activo', 'requiere_logistica',
    'requiere_revision_calidad', 'activa', 'es_proveedor_principal'
}
INTEGER_COLUMNS = {
    'fecha_id', 'anio', 'mes_numero', 'trimestre', 'semana_iso', 'dia_mes',
    'metros_cuadrados', 'vida_util_dias', 'lead_time_promedio_dias',
    'lead_time_dias', 'cantidad_minima_pedido', 'numero_lineas',
    'linea_numero', 'cantidad', 'cantidad_devuelta', 'impresiones', 'clics',
    'leads', 'conversiones_atribuidas', 'stock_inicial', 'entradas_compras',
    'salidas_ventas', 'ajuste_unidades', 'merma_unidades', 'stock_final',
    'punto_reorden', 'stock_seguridad', 'cobertura_dias', 'cantidad_comprada',
    'dias_prometidos', 'dias_reales_entrega', 'dias_retraso', 'filas_carga',
    'valores_nulos_o_blancos', 'errores_fk_detectados', 'duplicados_detectados',
    'ventas', 'devoluciones', 'meta_tickets', 'meta_nps', 'meta_entregas'
}
DOUBLE_COLUMNS = {'latitud', 'longitud', 'peso_estimado_kg', 'ticket_promedio'}
PRIMARY_KEYS = {
    'dim_fecha': ['fecha_id'],
    'dim_ciudad': ['ciudad_id'],
    'dim_tienda': ['tienda_id'],
    'dim_area_negocio': ['area_id'],
    'dim_centro_costo': ['centro_costo_id'],
    'dim_categoria_producto': ['categoria_id'],
    'dim_producto': ['producto_id'],
    'dim_proveedor': ['proveedor_id'],
    'bridge_producto_proveedor': ['producto_id', 'proveedor_id'],
    'dim_cliente': ['cliente_id'],
    'dim_empleado': ['empleado_id'],
    'dim_canal_venta': ['canal_id'],
    'dim_metodo_pago': ['metodo_pago_id'],
    'dim_campana_marketing': ['campana_id'],
    'dim_transportista': ['transportista_id'],
    'dim_motivo_devolucion': ['motivo_devolucion_id'],
    'fact_ventas_cabecera': ['venta_id'],
    'fact_ventas_detalle': ['detalle_venta_id'],
    'fact_devoluciones': ['devolucion_id'],
    'fact_marketing': ['marketing_id'],
    'fact_inventario_mensual': ['mes_carga', 'tienda_id', 'producto_id'],
    'fact_ajustes_inventario': ['ajuste_id'],
    'fact_compras': ['orden_compra_id'],
    'fact_logistica_entregas': ['entrega_id'],
    'fact_costos_mensual': ['costo_id'],
    'fact_presupuesto_mensual': ['presupuesto_id'],
    'fact_objetivos_mensuales': ['objetivo_id'],
    'fact_calidad_datos_carga': ['control_id'],
    'resumen_mensual_validacion': ['mes_carga'],
    'manifest': ['archivo'],
    'data_dictionary': ['tabla', 'campo'],
    'schema_relationships': ['tabla_origen', 'campo_origen', 'tabla_destino', 'campo_destino']
}
EXCLUDED_TABLES = {'bronze_file_load_audit'}

def bronze_tables():
    return [
        row.tableName
        for row in spark.sql(f'SHOW TABLES IN `{catalog}`.`{bronze_schema}`').collect()
        if not row.isTemporary and row.tableName not in EXCLUDED_TABLES
    ]

def clean_string_columns(df):
    for field in df.schema.fields:
        if field.dataType.simpleString() == 'string':
            cleaned = F.trim(F.col(field.name))
            df = df.withColumn(field.name, F.when(cleaned == '', F.lit(None)).otherwise(cleaned))
    return df

def cast_columns(df):
    for column in df.columns:
        if column in DATE_COLUMNS:
            df = df.withColumn(column, F.to_date(F.col(column)))
        elif column in BOOLEAN_COLUMNS:
            df = df.withColumn(column, F.col(column).cast('boolean'))
        elif column in DECIMAL_18_2_COLUMNS:
            df = df.withColumn(column, F.col(column).cast('decimal(18,2)'))
        elif column in DECIMAL_10_4_COLUMNS:
            df = df.withColumn(column, F.col(column).cast('decimal(10,4)'))
        elif column in INTEGER_COLUMNS:
            df = df.withColumn(column, F.col(column).cast('long'))
        elif column in DOUBLE_COLUMNS:
            df = df.withColumn(column, F.col(column).cast('double'))
        elif column.endswith('_id') or column in {'sku', 'moneda'}:
            df = df.withColumn(column, F.col(column).cast('string'))
    if 'mes_carga' in df.columns:
        df = df.withColumn('mes_carga', F.date_format(F.to_date(F.concat(F.col('mes_carga').cast('string'), F.lit('-01'))), 'yyyy-MM'))
    return df

def write_silver(df, table_name):
    target_table = f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    target_path = f'{silver_root}/{table_name}'
    (df.write.format('delta')
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .option('path', target_path)
       .saveAsTable(target_table))
    return {'table_name': table_name, 'target_table': target_table, 'target_path': target_path, 'rows': df.count()}

summary = []
for table_name in bronze_tables():
    source_table = f'`{catalog}`.`{bronze_schema}`.`{table_name}`'
    df = spark.table(source_table)
    df = clean_string_columns(df)
    df = cast_columns(df)
    keys = [key for key in PRIMARY_KEYS.get(table_name, []) if key in df.columns]
    if keys:
        df = df.dropDuplicates(keys)
    df = (df.withColumn('_np_silver_domain', F.lit(domain))
            .withColumn('_np_silver_layer', F.lit('silver'))
            .withColumn('_np_silver_step', F.lit('schema_standardization'))
            .withColumn('_np_silver_processed_ts', F.current_timestamp()))
    summary.append(write_silver(df, table_name))

display(spark.createDataFrame(summary))
